In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install pyspark vaderSentiment -q

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import FloatType
from pyspark.ml.feature import VectorAssembler, StandardScaler, Tokenizer, StopWordsRemover, HashingTF, IDF, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

spark = (
    SparkSession.builder
    .appName("FakeReviewDetection")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)

# Load cleaned dataset
reviews = spark.read.parquet("/content/drive/MyDrive/reviews_cleaned.parquet")
print(f"Total reviews: {reviews.count():,}")
reviews.printSchema()

In [ ]:
print("=" * 60)
print("LAYER 1: User Behavior Clustering")
print("=" * 60)

# Aggregate user-level features
user_features = reviews.groupBy("user_id").agg(
    F.count("*").alias("review_count"),
    F.countDistinct("parent_asin").alias("unique_products"),
    F.avg("rating").alias("avg_rating"),
    F.stddev("rating").alias("rating_std"),
    F.avg(F.when(F.col("rating") == 5.0, 1).otherwise(0)).alias("pct_5_star"),
    F.avg(F.when(F.col("rating") == 1.0, 1).otherwise(0)).alias("pct_1_star"),
    F.avg(F.when((F.col("rating") == 5.0) | (F.col("rating") == 1.0), 1).otherwise(0)).alias("pct_extreme"),
    F.avg("review_len").alias("avg_review_length"),
    F.stddev("review_len").alias("review_length_std"),
    F.avg("helpful_vote").alias("avg_helpful_votes"),
    F.sum("helpful_vote").alias("total_helpful_votes"),
    F.avg("verified_int").alias("pct_verified"),
    F.avg("has_image").alias("pct_with_images"),
    F.min("review_date").alias("first_review_date"),
    F.max("review_date").alias("last_review_date"),
    F.countDistinct("review_date").alias("active_days")
)

# Calculate review frequency
user_features = user_features.withColumn(
    "days_span",
    F.datediff(F.col("last_review_date"), F.col("first_review_date")) + 1
).withColumn(
    "review_frequency",
    F.when(F.col("days_span") > 0, F.col("review_count") / F.col("days_span")).otherwise(F.col("review_count"))
)

# Fill nulls
user_features = user_features.fillna({
    "rating_std": 0.0, "review_length_std": 0.0, "avg_helpful_votes": 0.0,
    "total_helpful_votes": 0, "pct_with_images": 0.0
})

user_features.cache()
print(f"Total unique users: {user_features.count():,}")

# Filter active users (>= 3 reviews)
active_users = user_features.filter(F.col("review_count") >= 3)
print(f"Active users (>=3 reviews): {active_users.count():,}")

In [ ]:
# Select features for clustering
cluster_features = [
    "review_count", "avg_rating", "rating_std", "pct_5_star", "pct_1_star",
    "pct_extreme", "avg_review_length", "avg_helpful_votes", "pct_verified", "review_frequency"
]

# Build ML Pipeline
assembler = VectorAssembler(inputCols=cluster_features, outputCol="features_raw", handleInvalid="skip")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)
prep_pipeline = Pipeline(stages=[assembler, scaler])
prep_model = prep_pipeline.fit(active_users)
user_data_scaled = prep_model.transform(active_users)

# Find Optimal K
print("\n--- Finding Optimal K ---")
k_values = range(2, 11)
wssse_values = []
silhouette_values = []
evaluator = ClusteringEvaluator(predictionCol="cluster", featuresCol="features", metricName="silhouette")

for k in k_values:
    kmeans = KMeans(k=k, seed=42, featuresCol="features", predictionCol="cluster")
    model = kmeans.fit(user_data_scaled)
    wssse = model.summary.trainingCost
    wssse_values.append(wssse)
    predictions = model.transform(user_data_scaled)
    silhouette = evaluator.evaluate(predictions)
    silhouette_values.append(silhouette)
    print(f"K={k}: WSSSE={wssse:,.0f}, Silhouette={silhouette:.4f}")

optimal_k = list(k_values)[np.argmax(silhouette_values)]
print(f"\n*** Optimal K based on Silhouette: {optimal_k} ***")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Elbow plot
axes[0].plot(list(k_values), wssse_values, 'bo-', linewidth=2)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('WSSSE')
axes[0].set_title('Elbow Method')
axes[0].grid(True)

# Silhouette plot
axes[1].plot(list(k_values), silhouette_values, 'ro-', linewidth=2)
axes[1].axvline(x=optimal_k, color='green', linestyle='--', label=f'Optimal K={optimal_k}')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=300)
plt.show()

In [ ]:
# Train Final K-Means Model
print(f"\n--- Training Final K-Means with K={optimal_k} ---")
kmeans_final = KMeans(k=optimal_k, seed=42, featuresCol="features", predictionCol="user_cluster", maxIter=100)
kmeans_model = kmeans_final.fit(user_data_scaled)
user_clustered = kmeans_model.transform(user_data_scaled)

# Analyze clusters
cluster_stats = user_clustered.groupBy("user_cluster").agg(
    F.count("*").alias("user_count"),
    F.avg("review_count").alias("avg_reviews"),
    F.avg("avg_rating").alias("avg_rating"),
    F.avg("pct_5_star").alias("avg_5star_pct"),
    F.avg("pct_1_star").alias("avg_1star_pct"),
    F.avg("pct_verified").alias("avg_verified_pct"),
    F.avg("avg_review_length").alias("avg_review_len"),
    F.avg("review_frequency").alias("avg_frequency"),
    F.avg("avg_helpful_votes").alias("avg_helpful")
).orderBy("user_cluster")

print("\nCluster Statistics:")
cluster_stats.show()

# Convert to pandas for visualization
cluster_pdf = cluster_stats.toPandas()

In [ ]:
# Cluster heatmap
heatmap_cols = ['avg_reviews', 'avg_rating', 'avg_5star_pct', 'avg_verified_pct',
                'avg_review_len', 'avg_frequency', 'avg_helpful']
heatmap_data = cluster_pdf.set_index('user_cluster')[heatmap_cols]

# Normalize for heatmap
heatmap_normalized = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_normalized, annot=True, cmap='RdYlGn', fmt='.2f', linewidths=0.5)
plt.title('User Cluster Behavior Heatmap (Normalized)')
plt.xlabel('Features')
plt.ylabel('Cluster')
plt.tight_layout()
plt.savefig('cluster_heatmap.png', dpi=300)
plt.show()

In [ ]:
# PCA 2D Cluster Visualization
from pyspark.ml.feature import PCA
from pyspark.sql.types import FloatType
from pyspark.sql.functions import udf

print("Running PCA for 2D visualization...")

# Reduce to 2D using PCA
pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(user_data_scaled)
user_pca = pca_model.transform(user_clustered)

# Extract PCA coordinates
@udf(FloatType())
def get_pc1(v):
    return float(v[0])

@udf(FloatType())
def get_pc2(v):
    return float(v[1])

user_pca = user_pca.withColumn("pc1", get_pc1(F.col("pca_features"))) \
                   .withColumn("pc2", get_pc2(F.col("pca_features")))

# Sample for visualization
plot_data = user_pca.select("pc1", "pc2", "user_cluster", "is_suspicious_cluster") \
    .sample(fraction=0.5, seed=42).toPandas()

# Scatter plot
plt.figure(figsize=(12, 8))

for i in range(optimal_k):
    cluster_data = plot_data[plot_data['user_cluster'] == i]
    plt.scatter(cluster_data['pc1'], cluster_data['pc2'],
                label=f'Cluster {i} (n={len(cluster_data)})',
                alpha=0.6, s=20)

plt.xlabel('Principal Component 1', fontsize=12)
plt.ylabel('Principal Component 2', fontsize=12)
plt.title('User Behavior Clusters (PCA 2D Projection)', fontsize=14)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cluster_pca_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

# Print explained variance
print(f"\nPCA Explained Variance:")
print(f"  PC1: {pca_model.explainedVariance[0]:.2%}")
print(f"  PC2: {pca_model.explainedVariance[1]:.2%}")
print(f"  Total: {sum(pca_model.explainedVariance):.2%}")

In [ ]:
# Identify Suspicious Clusters
print("\n--- Identifying Suspicious Clusters ---")
suspicious_clusters = cluster_stats.filter(
    ((F.col("avg_5star_pct") > 0.7) & (F.col("avg_verified_pct") < 0.5)) |
    ((F.col("avg_frequency") > 0.3) & (F.col("avg_review_len") < 150)) |
    ((F.col("avg_5star_pct") > 0.85) & (F.col("avg_helpful") < 0.3))
).select("user_cluster")

suspicious_cluster_ids = [row.user_cluster for row in suspicious_clusters.collect()]
print(f"Suspicious Cluster IDs: {suspicious_cluster_ids}")

user_clustered = user_clustered.withColumn(
    "is_suspicious_cluster",
    F.when(F.col("user_cluster").isin(suspicious_cluster_ids), 1).otherwise(0)
)

suspicious_user_count = user_clustered.filter(F.col("is_suspicious_cluster") == 1).count()
total_clustered = user_clustered.count()
print(f"Suspicious users from clustering: {suspicious_user_count:,} ({suspicious_user_count/total_clustered*100:.2f}%)")

user_clustered.cache()

In [ ]:
print("\n" + "=" * 60)
print("LAYER 2: Sentiment-Rating Mismatch Detection")
print("=" * 60)

from pyspark.sql.functions import udf
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

@udf(FloatType())
def get_sentiment(text):
    if text is None or len(str(text).strip()) == 0:
        return 0.0
    try:
        scores = analyzer.polarity_scores(str(text))
        return float(scores['compound'])
    except:
        return 0.0

# Sample for sentiment analysis (30% to save time)
reviews_sample = reviews.sample(fraction=0.3, seed=42)
reviews_sentiment = reviews_sample.withColumn("sentiment_score", get_sentiment(F.col("text")))
print(f"Sentiment analysis on {reviews_sentiment.count():,} reviews")

# Calculate rating-sentiment gap
reviews_sentiment = reviews_sentiment.withColumn(
    "rating_normalized", (F.col("rating") - 3) / 2
).withColumn(
    "rating_sentiment_gap", F.abs(F.col("rating_normalized") - F.col("sentiment_score"))
).withColumn(
    "suspicious_sentiment",
    F.when((F.col("rating") == 5.0) & (F.col("sentiment_score") < -0.1), 1)
    .when((F.col("rating") == 1.0) & (F.col("sentiment_score") > 0.3), 1)
    .otherwise(0)
)

suspicious_sentiment_count = reviews_sentiment.filter(F.col("suspicious_sentiment") == 1).count()
total_sentiment = reviews_sentiment.count()
print(f"Reviews with suspicious sentiment: {suspicious_sentiment_count:,} ({suspicious_sentiment_count/total_sentiment*100:.2f}%)")

reviews_sentiment.cache()

In [ ]:
# Sentiment vs Rating scatter plot
sentiment_pdf = reviews_sentiment.select("rating", "sentiment_score", "suspicious_sentiment").toPandas()

plt.figure(figsize=(10, 6))
colors = ['blue' if s == 0 else 'red' for s in sentiment_pdf['suspicious_sentiment']]
plt.scatter(sentiment_pdf['rating'], sentiment_pdf['sentiment_score'], c=colors, alpha=0.3, s=10)
plt.xlabel('Rating')
plt.ylabel('Sentiment Score (VADER)')
plt.title('Rating vs Sentiment Score (Red = Suspicious Mismatch)')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.grid(True, alpha=0.3)
plt.savefig('sentiment_rating_scatter.png', dpi=300)
plt.show()

## Cell 10: Layer 3 - Text Similarity Detection

In [ ]:
print("\n" + "=" * 60)
print("LAYER 3: Text Similarity Detection (TF-IDF)")
print("=" * 60)

from pyspark.ml.feature import Normalizer
from pyspark.ml.functions import vector_to_array
from itertools import combinations

# Get users with multiple reviews
user_review_counts = reviews.groupBy("user_id").count().filter(F.col("count") >= 3)
reviews_multi = reviews.join(user_review_counts, on="user_id", how="inner")

# Sample users for TF-IDF analysis (10% to save memory)
sampled_users = reviews_multi.select("user_id").distinct().sample(fraction=0.1, seed=42)
reviews_tfidf = reviews_multi.join(sampled_users, on="user_id", how="inner")
print(f"Reviews for TF-IDF analysis: {reviews_tfidf.count():,}")

# TF-IDF Pipeline
tokenizer = Tokenizer(inputCol="text", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashingTF = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=1000)
idf = IDF(inputCol="raw_features", outputCol="tfidf_features")
normalizer = Normalizer(inputCol="tfidf_features", outputCol="norm_features", p=2.0)

tfidf_pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, normalizer])
tfidf_model = tfidf_pipeline.fit(reviews_tfidf)
reviews_tfidf_transformed = tfidf_model.transform(reviews_tfidf)

# Convert to array for similarity calculation
reviews_tfidf_transformed = reviews_tfidf_transformed.withColumn(
    "features_array", vector_to_array(F.col("norm_features"))
)

# Calculate cosine similarity between reviews of same user
# Group reviews by user and collect features
user_reviews_collected = reviews_tfidf_transformed.groupBy("user_id").agg(
    F.collect_list("features_array").alias("all_features"),
    F.count("*").alias("review_count")
).filter(F.col("review_count") >= 2)

# UDF to calculate average pairwise cosine similarity
from pyspark.sql.types import FloatType
import numpy as np

@udf(FloatType())
def calc_avg_similarity(features_list):
    if features_list is None or len(features_list) < 2:
        return 0.0
    try:
        similarities = []
        for i in range(len(features_list)):
            for j in range(i + 1, len(features_list)):
                vec1 = np.array(features_list[i])
                vec2 = np.array(features_list[j])
                # Cosine similarity (vectors are already normalized)
                sim = np.dot(vec1, vec2)
                similarities.append(sim)
        return float(np.mean(similarities)) if similarities else 0.0
    except:
        return 0.0

user_similarity = user_reviews_collected.withColumn(
    "avg_text_similarity", calc_avg_similarity(F.col("all_features"))
)

# Define threshold for suspicious similarity
high_similarity_threshold = 0.5

# Flag users with high text similarity
user_similarity = user_similarity.withColumn(
    "is_high_similarity",
    F.when(F.col("avg_text_similarity") > high_similarity_threshold, 1).otherwise(0)
)

high_sim_users = user_similarity.filter(F.col("is_high_similarity") == 1).count()
total_sim_users = user_similarity.count()
print(f"Users analyzed: {total_sim_users:,}")
print(f"Users with high text similarity (>{high_similarity_threshold}): {high_sim_users:,} ({high_sim_users/total_sim_users*100:.2f}%)")

user_similarity.cache()

In [ ]:
# Text Similarity Visualization
similarity_pd = user_similarity.select("avg_text_similarity", "review_count").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of text similarity
axes[0].hist(similarity_pd['avg_text_similarity'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=high_similarity_threshold, color='r', linestyle='--', label=f'Threshold={high_similarity_threshold}')
axes[0].set_xlabel('Average Text Similarity', fontsize=12)
axes[0].set_ylabel('Number of Users', fontsize=12)
axes[0].set_title('Distribution of User Text Similarity', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot: similarity vs review count
axes[1].scatter(similarity_pd['review_count'], similarity_pd['avg_text_similarity'], alpha=0.3, s=10)
axes[1].axhline(y=high_similarity_threshold, color='r', linestyle='--')
axes[1].set_xlabel('Review Count', fontsize=12)
axes[1].set_ylabel('Average Text Similarity', fontsize=12)
axes[1].set_title('Text Similarity vs Review Count', fontsize=14)
axes[1].set_xlim(0, 50)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('text_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 11: Layer 4 - Review Burst Detection

In [ ]:
print("\n" + "=" * 60)
print("LAYER 4: Review Burst Detection")
print("=" * 60)

# Count reviews per product per day
daily_reviews = reviews.groupBy("parent_asin", "review_date").agg(
    F.count("*").alias("daily_count")
)

# Define burst threshold (>=3 reviews in one day)
burst_threshold = 3

daily_reviews = daily_reviews.withColumn(
    "is_burst",
    F.when(F.col("daily_count") >= burst_threshold, 1).otherwise(0)
)

# Products with burst patterns
burst_products = daily_reviews.filter(F.col("is_burst") == 1).select("parent_asin").distinct()
burst_count = burst_products.count()
total_products = daily_reviews.select("parent_asin").distinct().count()
print(f"Products with review burst patterns: {burst_count:,} ({burst_count/total_products*100:.2f}%)")

# Count total burst events
burst_events = daily_reviews.filter(F.col("is_burst") == 1).count()
print(f"Total burst events (days): {burst_events:,}")

# Top burst events
print("\nTop 10 Burst Events:")
daily_reviews.filter(F.col("is_burst") == 1).orderBy(F.desc("daily_count")).show(10)

daily_reviews.cache()

In [ ]:
# Review Burst Visualization
burst_sample = daily_reviews.filter(F.col("is_burst") == 1).orderBy(F.desc("daily_count")).first()

if burst_sample:
    sample_product = burst_sample.parent_asin
    product_daily = daily_reviews.filter(F.col("parent_asin") == sample_product).orderBy("review_date").toPandas()

    plt.figure(figsize=(14, 5))

    # Plot daily reviews
    plt.bar(product_daily['review_date'], product_daily['daily_count'],
            color='steelblue', alpha=0.7, label='Daily Reviews')

    # Highlight burst days
    burst_dates = product_daily[product_daily['is_burst'] == 1]
    plt.bar(burst_dates['review_date'], burst_dates['daily_count'],
            color='red', alpha=0.9, label='Burst Detected')

    # Add threshold line
    plt.axhline(y=burst_threshold, color='red', linestyle='--', alpha=0.5, label=f'Burst Threshold (>={burst_threshold})')

    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Daily Review Count', fontsize=12)
    plt.title(f'Review Burst Detection - Product: {sample_product}', fontsize=14)
    plt.legend()
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('review_burst.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Product {sample_product}: {len(burst_dates)} burst day(s) detected")
else:
    print("No burst events found in dataset.")

## Cell 12: Layer 5 - Rating Entropy Analysis

In [ ]:
print("\n" + "=" * 60)
print("LAYER 5: Rating Distribution Entropy")
print("=" * 60)

# Calculate rating distribution per product
rating_counts = reviews.groupBy("parent_asin", "rating").count()

# Pivot to get columns for each rating
rating_dist = rating_counts.groupBy("parent_asin").pivot("rating").sum("count").fillna(0)

# Use backticks for column names with dots
rating_dist = rating_dist.withColumn(
    "total", F.col("`1.0`") + F.col("`2.0`") + F.col("`3.0`") + F.col("`4.0`") + F.col("`5.0`")
).filter(F.col("total") >= 10)

# Calculate proportions
rating_dist = rating_dist.withColumn("p1", F.col("`1.0`") / F.col("total")) \
                         .withColumn("p2", F.col("`2.0`") / F.col("total")) \
                         .withColumn("p3", F.col("`3.0`") / F.col("total")) \
                         .withColumn("p4", F.col("`4.0`") / F.col("total")) \
                         .withColumn("p5", F.col("`5.0`") / F.col("total"))

# Calculate entropy: -sum(p * log2(p))
rating_dist = rating_dist.withColumn(
    "entropy",
    -1 * (
        F.when(F.col("p1") > 0, F.col("p1") * (F.log(F.col("p1")) / F.log(F.lit(2)))).otherwise(0) +
        F.when(F.col("p2") > 0, F.col("p2") * (F.log(F.col("p2")) / F.log(F.lit(2)))).otherwise(0) +
        F.when(F.col("p3") > 0, F.col("p3") * (F.log(F.col("p3")) / F.log(F.lit(2)))).otherwise(0) +
        F.when(F.col("p4") > 0, F.col("p4") * (F.log(F.col("p4")) / F.log(F.lit(2)))).otherwise(0) +
        F.when(F.col("p5") > 0, F.col("p5") * (F.log(F.col("p5")) / F.log(F.lit(2)))).otherwise(0)
    )
)

# Low entropy + high 5-star = suspicious
suspicious_entropy = rating_dist.filter(
    (F.col("entropy") < 0.5) & (F.col("p5") > 0.8)
)
suspicious_entropy_count = suspicious_entropy.count()
total_products = rating_dist.count()
print(f"Products analyzed: {total_products:,}")
print(f"Products with suspicious rating entropy: {suspicious_entropy_count:,} ({suspicious_entropy_count/total_products*100:.2f}%)")

# Show examples
print("\nTop suspicious products (low entropy, high 5-star):")
suspicious_entropy.select("parent_asin", "total", "p5", "entropy").orderBy("entropy").show(10)

rating_dist.cache()

In [ ]:
# Entropy Visualization
entropy_pd = rating_dist.filter(F.col("total") >= 5) \
    .withColumn("suspicious_distribution",
                F.when((F.col("entropy") < 0.5) & (F.col("p5") > 0.8), 1).otherwise(0)) \
    .select("entropy", "p5", "total", "suspicious_distribution") \
    .sample(fraction=0.2, seed=42).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of entropy
axes[0].hist(entropy_pd['entropy'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0.5, color='r', linestyle='--', label='Suspicious Threshold')
axes[0].set_xlabel('Rating Entropy', fontsize=12)
axes[0].set_ylabel('Number of Products', fontsize=12)
axes[0].set_title('Distribution of Product Rating Entropy', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter: entropy vs 5-star percentage
colors = ['red' if x == 1 else 'blue' for x in entropy_pd['suspicious_distribution']]
axes[1].scatter(entropy_pd['p5'], entropy_pd['entropy'], c=colors, alpha=0.3, s=10)
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
axes[1].axvline(x=0.8, color='r', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Percentage of 5-Star Reviews', fontsize=12)
axes[1].set_ylabel('Rating Entropy', fontsize=12)
axes[1].set_title('Entropy vs 5-Star Percentage\n(Red = Suspicious)', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('entropy_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 13: Layer 6 - Comprehensive Trust Index

In [ ]:
print("\n" + "=" * 60)
print("LAYER 6: Comprehensive Trust Index")
print("=" * 60)

# ===== USER TRUST SCORE =====
# Aggregate user-level suspicion signals
user_signals = user_clustered.select(
    "user_id", "user_cluster", "is_suspicious_cluster",
    "pct_5_star", "pct_verified", "avg_review_length", "review_frequency", "avg_helpful_votes"
)

# Sentiment aggregation by user
sentiment_by_user = reviews_sentiment.groupBy("user_id").agg(
    F.avg("suspicious_sentiment").alias("pct_suspicious_sentiment"),
    F.avg("rating_sentiment_gap").alias("avg_sentiment_gap")
)

# Join signals
user_trust_final = user_signals.join(sentiment_by_user, on="user_id", how="left").fillna({
    "pct_suspicious_sentiment": 0.0, "avg_sentiment_gap": 0.0
})

# Calculate User Trust Score (0-1, higher = more trustworthy)
user_trust_final = user_trust_final.withColumn(
    "user_trust_score",
    (
        0.25 * F.col("pct_verified") +
        0.20 * (1 - F.col("is_suspicious_cluster")) +
        0.15 * F.least(F.col("avg_review_length") / 500, F.lit(1.0)) +
        0.15 * (1 - F.col("pct_suspicious_sentiment")) +
        0.15 * F.least(F.col("avg_helpful_votes") / 5, F.lit(1.0)) +
        0.10 * (1 - F.least(F.col("review_frequency"), F.lit(1.0)))
    )
)

print("User Trust Score Statistics:")
user_trust_final.select("user_trust_score").describe().show()

user_trust_final.cache()

# ===== REVIEW TRUST SCORE =====
# Join user trust to reviews
review_trust_final = reviews.join(
    user_trust_final.select("user_id", "user_trust_score"),
    on="user_id",
    how="left"
).fillna({"user_trust_score": 0.5})

# Calculate Review Trust Score
review_trust_final = review_trust_final.withColumn(
    "review_trust_score",
    (
        0.40 * F.col("user_trust_score") +
        0.25 * F.col("verified_int") +
        0.20 * F.least(F.col("review_len") / 500, F.lit(1.0)) +
        0.15 * F.least(F.col("helpful_vote") / 5, F.lit(1.0))
    )
)

print("\nReview Trust Score Statistics:")
review_trust_final.select("review_trust_score").describe().show()

# Trust level distribution
review_trust_final = review_trust_final.withColumn(
    "trust_level",
    F.when(F.col("review_trust_score") >= 0.7, "High")
    .when(F.col("review_trust_score") >= 0.4, "Medium")
    .otherwise("Low")
)

print("\nReview Trust Level Distribution:")
review_trust_final.groupBy("trust_level").count().orderBy("trust_level").show()

review_trust_final.cache()

## Cell 14: Layer 6 - Trust Index Visualization

In [ ]:
# Trust Index Distribution
trust_pdf = user_trust.select("trust_index", "trust_level").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(trust_pdf['trust_index'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=trust_pdf['trust_index'].mean(), color='red', linestyle='--', label=f"Mean={trust_pdf['trust_index'].mean():.2f}")
axes[0].set_xlabel('Trust Index')
axes[0].set_ylabel('Count')
axes[0].set_title('User Trust Index Distribution')
axes[0].legend()

# Pie chart
trust_counts = trust_pdf['trust_level'].value_counts()
axes[1].pie(trust_counts, labels=trust_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Trust Level Distribution')

plt.tight_layout()
plt.savefig('trust_index_distribution.png', dpi=300)
plt.show()

In [ ]:
# Trust Index Visualization
user_trust_pd = user_trust_final.select("user_trust_score").sample(fraction=0.5, seed=42).toPandas()
review_trust_pd = review_trust_final.select("review_trust_score", "rating").sample(fraction=0.1, seed=42).toPandas()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# User Trust Score Distribution
axes[0].hist(user_trust_pd['user_trust_score'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=0.5, color='r', linestyle='--', label='Threshold=0.5')
axes[0].set_xlabel('User Trust Score', fontsize=12)
axes[0].set_ylabel('Number of Users', fontsize=12)
axes[0].set_title('User Trust Score Distribution', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Review Trust Score Distribution
axes[1].hist(review_trust_pd['review_trust_score'], bins=50, edgecolor='black', alpha=0.7, color='seagreen')
axes[1].axvline(x=0.5, color='r', linestyle='--', label='Threshold=0.5')
axes[1].set_xlabel('Review Trust Score', fontsize=12)
axes[1].set_ylabel('Number of Reviews', fontsize=12)
axes[1].set_title('Review Trust Score Distribution', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Review Trust by Rating
review_trust_pd.boxplot(column='review_trust_score', by='rating', ax=axes[2])
axes[2].set_xlabel('Rating', fontsize=12)
axes[2].set_ylabel('Review Trust Score', fontsize=12)
axes[2].set_title('Review Trust Score by Rating', fontsize=14)
plt.suptitle('')

plt.tight_layout()
plt.savefig('trust_index_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Model Evaluation with Visualization

In [ ]:
# ============================================================
# MODEL EVALUATION WITH VISUALIZATION
# ============================================================

print("\n" + "=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

# 1. Clustering Quality
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    predictionCol="user_cluster",
    featuresCol="features",
    metricName="silhouette"
)
silhouette = evaluator.evaluate(user_clustered)
print(f"Silhouette Score: {silhouette:.4f}")

# 2. Cluster Distribution Visualization
cluster_counts = user_clustered.groupBy("user_cluster").agg(
    F.count("*").alias("count"),
    F.sum("is_suspicious_cluster").alias("suspicious_count")
).orderBy("user_cluster").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart - Cluster sizes
colors = ['red' if cluster_counts.loc[i, 'suspicious_count'] > 0 else 'steelblue'
          for i in range(len(cluster_counts))]
axes[0].bar(cluster_counts['user_cluster'], cluster_counts['count'], color=colors, edgecolor='black')
axes[0].set_xlabel('Cluster ID', fontsize=12)
axes[0].set_ylabel('Number of Users', fontsize=12)
axes[0].set_title(f'Cluster Size Distribution\n(Red = Suspicious, Silhouette = {silhouette:.4f})', fontsize=14)
axes[0].grid(True, alpha=0.3, axis='y')

# Add count labels
for i, v in enumerate(cluster_counts['count']):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

# Pie chart - Suspicious vs Normal
suspicious_total = cluster_counts['suspicious_count'].sum()
normal_total = cluster_counts['count'].sum() - suspicious_total
axes[1].pie([normal_total, suspicious_total],
            labels=['Normal Users', 'Suspicious Users'],
            autopct='%1.1f%%', colors=['steelblue', 'red'],
            explode=[0, 0.1], startangle=90)
axes[1].set_title('User Classification Overview', fontsize=14)

plt.tight_layout()
plt.savefig('evaluation_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

# 3. Trust Index Distribution with Percentiles
trust_pd = review_trust_final.select("review_trust_score").toPandas()
percentiles = [0.1, 0.25, 0.5, 0.75, 0.9]
percentile_values = trust_pd['review_trust_score'].quantile(percentiles).values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with percentile lines
axes[0].hist(trust_pd['review_trust_score'], bins=50, edgecolor='black', alpha=0.7, color='seagreen')
colors_p = ['blue', 'green', 'orange', 'green', 'blue']
labels_p = ['10th', '25th', '50th (median)', '75th', '90th']
for pv, c, lb in zip(percentile_values, colors_p, labels_p):
    axes[0].axvline(x=pv, color=c, linestyle='--', linewidth=2, label=f'{lb}: {pv:.3f}')
axes[0].axvline(x=0.4, color='red', linestyle='-', linewidth=2, label='Suspicious Threshold (0.4)')
axes[0].set_xlabel('Review Trust Score', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Trust Score Distribution with Percentiles', fontsize=14)
axes[0].legend(loc='upper left', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Box plot by rating
review_trust_box = review_trust_final.select("review_trust_score", "rating").sample(fraction=0.1, seed=42).toPandas()
review_trust_box.boxplot(column='review_trust_score', by='rating', ax=axes[1])
axes[1].axhline(y=0.4, color='red', linestyle='--', label='Threshold')
axes[1].set_xlabel('Rating', fontsize=12)
axes[1].set_ylabel('Trust Score', fontsize=12)
axes[1].set_title('Trust Score by Rating', fontsize=14)
plt.suptitle('')

plt.tight_layout()
plt.savefig('evaluation_trust_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Print percentiles
print("\nTrust Score Percentiles:")
for p, v in zip(['10th', '25th', '50th', '75th', '90th'], percentile_values):
    print(f"  {p}: {v:.3f}")

# 4. Calculate detection counts for summary
high_sim_count = user_similarity.filter(F.col("is_high_similarity") == 1).count()
low_trust_reviews = review_trust_final.filter(F.col("review_trust_score") < 0.4).count()

# Detection Rates by Layer - Bar Chart
detection_data = {
    'Layer': ['L1: Suspicious\nClusters', 'L2: Sentiment\nMismatch', 'L3: Text\nSimilarity',
              'L4: Burst\nProducts', 'L5: Rating\nDistribution', 'L6: Low Trust\nReviews'],
    'Count': [suspicious_user_count, suspicious_sentiment_count, high_sim_count,
              burst_count, suspicious_entropy_count, low_trust_reviews],
    'Type': ['Users', 'Reviews', 'Users', 'Products', 'Products', 'Reviews']
}
detection_df = pd.DataFrame(detection_data)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(detection_df['Layer'], detection_df['Count'],
              color=['#e74c3c', '#3498db', '#e74c3c', '#2ecc71', '#2ecc71', '#3498db'],
              edgecolor='black')

# Add value labels
for bar, count in zip(bars, detection_df['Count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count:,}', ha='center', fontsize=10, fontweight='bold')

ax.set_ylabel('Count', fontsize=12)
ax.set_title('Detection Results by Layer\n(Red=Users, Blue=Reviews, Green=Products)', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('evaluation_detection_layers.png', dpi=150, bbox_inches='tight')
plt.show()

# 5. Confusion Matrix Style - Trust Score Classification
low_trust = trust_pd[trust_pd['review_trust_score'] < 0.4].shape[0]
medium_trust = trust_pd[(trust_pd['review_trust_score'] >= 0.4) & (trust_pd['review_trust_score'] < 0.7)].shape[0]
high_trust = trust_pd[trust_pd['review_trust_score'] >= 0.7].shape[0]

fig, ax = plt.subplots(figsize=(8, 6))
categories = ['Low Trust\n(<0.4)\nSuspicious', 'Medium Trust\n(0.4-0.7)\nUncertain', 'High Trust\n(>=0.7)\nTrustworthy']
counts = [low_trust, medium_trust, high_trust]
colors = ['#e74c3c', '#f39c12', '#27ae60']

bars = ax.bar(categories, counts, color=colors, edgecolor='black', width=0.6)
for bar, count in zip(bars, counts):
    percentage = count / len(trust_pd) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{count:,}\n({percentage:.1f}%)', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('Number of Reviews', fontsize=12)
ax.set_title('Review Classification by Trust Score', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('evaluation_classification.png', dpi=150, bbox_inches='tight')
plt.show()

# 6. Summary Statistics
print("\n" + "=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print(f"""
Clustering Quality:
  - Silhouette Score: {silhouette:.4f} (Range: -1 to 1, higher = better)
  - Number of Clusters: {optimal_k}

Detection Coverage:
  - Total Users Analyzed: {user_clustered.count():,}
  - Total Reviews Analyzed: {review_trust_final.count():,}

Classification Results:
  - Low Trust (Suspicious): {low_trust:,} ({low_trust/len(trust_pd)*100:.1f}%)
  - Medium Trust (Uncertain): {medium_trust:,} ({medium_trust/len(trust_pd)*100:.1f}%)
  - High Trust (Trustworthy): {high_trust:,} ({high_trust/len(trust_pd)*100:.1f}%)
""")

## Cell 15: Final Summary

In [ ]:
print("\n" + "=" * 60)
print("FINAL SUMMARY: 6-Layer Fake Review Detection")
print("=" * 60)

print(f"""
Layer 1 - User Clustering:
  - Optimal K: {optimal_k}
  - Suspicious clusters: {suspicious_cluster_ids}
  - Suspicious users: {suspicious_user_count:,} ({suspicious_user_count/total_clustered*100:.2f}%)

Layer 2 - Sentiment Mismatch:
  - Analyzed reviews: {reviews_sentiment.count():,}
  - Suspicious reviews: {suspicious_sentiment_count:,} ({suspicious_sentiment_count/total_sentiment*100:.2f}%)

Layer 3 - Text Similarity:
  - Users analyzed: {user_text_stats.count():,}

Layer 4 - Review Burst:
  - Products with bursts: {burst_count:,}

Layer 5 - Rating Entropy:
  - Suspicious products: {suspicious_entropy_count:,}

Layer 6 - Trust Index:
  - Mean trust score: {trust_pdf['trust_index'].mean():.3f}
  - Low trust users: {(trust_pdf['trust_level'] == 'Low').sum():,}
""")

# Save final user trust scores
output_path = "/content/drive/MyDrive/user_trust_scores.parquet"
user_trust.write.mode("overwrite").parquet(output_path)
print(f"\nUser trust scores saved to: {output_path}")

In [ ]:
# Save reviews with trust scores
print("Saving final dataset with trust scores...")

output_path = "/content/drive/MyDrive/reviews_with_trust_score.parquet"
review_trust_final.write.mode("overwrite").parquet(output_path)
print(f"Dataset saved to: {output_path}")

# Verify
test_load = spark.read.parquet(output_path)
print(f"Saved rows: {test_load.count():,}")
print(f"Columns: {test_load.columns}")

In [ ]:
# Export Results
print("\n--- Exporting Results ---")

# Export suspicious users
suspicious_users_df = user_trust_final.filter(F.col("user_trust_score") < 0.4)
suspicious_users_pd = suspicious_users_df.toPandas()
suspicious_users_pd.to_csv('suspicious_users.csv', index=False)
print(f"Exported: suspicious_users.csv ({len(suspicious_users_pd):,} rows)")

# Export low trust reviews (limit to 10000 for file size)
low_trust_reviews_df = review_trust_final.filter(F.col("review_trust_score") < 0.4)
low_trust_reviews_pd = low_trust_reviews_df.limit(10000).toPandas()
low_trust_reviews_pd.to_csv('low_trust_reviews.csv', index=False)
print(f"Exported: low_trust_reviews.csv ({len(low_trust_reviews_pd):,} rows)")

# Export cluster statistics
cluster_pdf.to_csv('cluster_statistics.csv', index=False)
print("Exported: cluster_statistics.csv")

print("\n" + "=" * 80)

 ## Export Results

In [ ]:
# Export results for Streamlit Demo
print("Exporting results to Google Drive...")

# Save to Google Drive
output_path = "/content/drive/MyDrive/fake_review_demo/"
!mkdir -p {output_path}

# Export CSVs
suspicious_users_pd.to_csv(f'{output_path}suspicious_users.csv', index=False)
low_trust_reviews_pd.to_csv(f'{output_path}low_trust_reviews.csv', index=False)
cluster_pdf.to_csv(f'{output_path}cluster_statistics.csv', index=False)

# Copy images
!cp *.png {output_path}

print(f"All files exported to: {output_path}")
!ls -la {output_path}

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)
print("""
Generated Files:
  Visualizations:
  - elbow_silhouette.png
  - cluster_heatmap.png
  - cluster_pca_scatter.png
  - sentiment_rating_scatter.png
  - text_similarity.png
  - review_burst.png
  - entropy_analysis.png
  - trust_index_distribution.png
  - evaluation_clusters.png
  - evaluation_trust_distribution.png
  - evaluation_detection_layers.png
  - evaluation_classification.png

  Data Exports:
  - suspicious_users.csv
  - low_trust_reviews.csv
  - cluster_statistics.csv
  - reviews_with_trust_score.parquet (in Drive root)
""")